# 06 · Feature engineering, round 2

**Goal:** raise AUC beyond the tuned CatBoost (15-fold CV **0.7117**, public leaderboard 0.723; leader 0.738).

**Approach**
1. **Mine the call notes for rules.** Notes exist only for training quotes and can't be features, but each "why not bound" reason may be triggered by a rule on the structured columns. Recovering those rules could expose hidden signal (as `DRIVERS_PER_VEH`, the last big win, did).
2. **Build a broad candidate library** (67 target-free features, `features.add_v2`) from the raw driver and vehicle tables, price benchmarks, frequencies and dates.
3. **Screen against the residuals** of the current model's OOF predictions: a feature the model already uses fully shows no residual pattern.
4. **Test groups in tuned CatBoost** on the seed-0 folds (paired Δ vs. the same folds), then confirm anything promising on all 15 folds.
5. **Pruning:** the tuned model may now prefer *fewer* features; drop-group tests with the tuned parameters.

In [1]:
import json, sys, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
sys.path.insert(0, '.')
import common as C, features as FE

lab, hold = C.load()
L = FE.add_v2(FE.add_static(lab), FE.add_static(pd.concat([lab, hold])))
T = json.load(open('params_tuned.json'))['cb']
F, P = T['features'], T['params']
V = FE.v2_columns(L)
base = C.cv_score('probe_base', L, F, 'cb', P, seeds=(0,), log=False)
fmt = {'auc': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}'}

def test(name, feats, seeds=(0,)):
    r = C.cv_score(name, L, feats, 'cb', P, seeds=seeds, log=False)
    b = base if seeds == (0,) else C.cv_score('fe2_base15', L, F, 'cb', P)
    d = r['fold_aucs'] - b['fold_aucs']
    return {'auc': r['auc_mean'], 'Δ auc': d.mean(), 'se Δ': d.std() / np.sqrt(len(d))}
print(f'{len(V)} candidate features | tuned CatBoost seed-0 AUC {base["auc_mean"]:.4f}')

67 candidate features | tuned CatBoost seed-0 AUC 0.7130


/Users/joerussell/Desktop/PROJECTS/competitions/travelers-umc/MODELS/features.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['V2_DAY_OF_YEAR'] = dt.dt.dayofyear
/Users/joerussell/Desktop/PROJECTS/competitions/travelers-umc/MODELS/features.py:184: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['V2_DAY_OF_MONTH'] = dt.dt.day
/Users/joerussell/Desktop/PROJECTS/competitions/travelers-umc/MODELS/features.py:185: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

## 1. Call notes: what triggers each "why not bound" reason?
Each note's reason clauses are mapped to 10 categories by keyword. For each category: how often it appears among converted vs. lost quotes, and the best single-split rule (depth-1 tree) on the structured columns that predicts it.

In [2]:
cn = pd.read_parquet('../data/processed/call_notes.parquet')
w = cn.CALL_NOTE.str.extract(r'Why not bound:(.*)$')[0].str.lower()
CATS = {'price': r"budget|willing to pay|competitive|cheaper|too steep|more than expected|tradeoff|monthly cost|price point|higher than expected",
        'no discount': r"discount(s)? (applied|available)|absence of discounts|without any discounts|not enough discounts",
        'home bundle': r"bundl|home policy|home carrier", 'lienholder': r"lienholder", 'lease': r"leas|lessor",
        'low package': r"low package", 'high package': r"high package|leaner|stripped down", 'other drivers': r"drivers",
        'CAT zone': r"cat zone|storm|cat exposure", 'unclear': r"unclear|couldn't resolve|follow-up|open questions|unresolved"}
R = pd.DataFrame({k: w.str.contains(v, regex=True).astype(int) for k, v in CATS.items()}).assign(POLICY_ID=cn.POLICY_ID, y=cn.CONVERT_IND)
d = R.merge(L.drop(columns=C.TARGET), on='POLICY_ID')
X = d[F + ['NUM_LOANED_VEH', 'NUM_LEASED_VEH', 'TOTAL_NUM_VEH']].copy()
X = X.assign(**{c: X[c].cat.codes for c in X.select_dtypes('category').columns}).fillna(-1)
rows = []
for k in CATS:
    t = DecisionTreeClassifier(max_depth=1, min_samples_leaf=50).fit(X, d[k])
    f, thr = X.columns[t.tree_.feature[0]], t.tree_.threshold[0]
    hi = d[k][X[f] > thr].mean(); lo = d[k][X[f] <= thr].mean()
    rows.append({'reason': k, 'share (lost)': d.loc[d.y == 0, k].mean(), 'share (converted)': d.loc[d.y == 1, k].mean(),
                 'trigger': f'{f} > {thr:.1f}', 'share if trigger': hi, 'share otherwise': lo})
print(f'{len(cn):,} notes; {cn.CONVERT_IND.mean():.1%} of noted quotes converted; every note has a "why not bound" clause')
pd.DataFrame(rows).set_index('reason').style.format('{:.1%}', subset=['share (lost)', 'share (converted)', 'share if trigger', 'share otherwise'])

/var/folders/z4/v6bydh_x01dffn_c76yl_w1w0000gn/T/ipykernel_52633/666679265.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  R = pd.DataFrame({k: w.str.contains(v, regex=True).astype(int) for k, v in CATS.items()}).assign(POLICY_ID=cn.POLICY_ID, y=cn.CONVERT_IND)


15,184 notes; 12.4% of noted quotes converted; every note has a "why not bound" clause


,share (lost),share (converted),trigger,share if trigger,share otherwise
reason,,,,,
price,52.2%,51.6%,DISCOUNT_IND > 0.5,29.2%,60.7%
no discount,17.3%,16.8%,DISCOUNT_IND > 0.5,0.0%,23.7%
home bundle,23.2%,22.0%,NUM_LOANED_VEH > 0.5,20.5%,33.1%
lienholder,15.9%,18.3%,NUM_LOANED_VEH > 0.5,20.2%,0.0%
lease,7.5%,8.3%,NUM_LEASED_VEH > 0.5,19.1%,0.0%
low package,7.1%,9.6%,COV_PKG_TYPE > 1.5,0.0%,19.9%
high package,3.4%,2.6%,COV_PKG_TYPE > 2.5,16.4%,0.0%
other drivers,6.3%,4.4%,NUM_DRIVERS > 2.5,18.0%,0.0%
CAT zone,5.3%,3.5%,CAT_ZONE > 3.5,17.7%,0.0%


**Reading:** the notes are decoration, not hidden signal.
- Every reason maps to a **simple threshold on one policy column**: lienholder ↔ a loaned vehicle (policy-table `NUM_LOANED_VEH`), lease ↔ a leased vehicle, other drivers ↔ 3+ drivers, CAT zone ↔ zone ≥ 4, low/high package ↔ `COV_PKG_TYPE`, no discount ↔ `DISCOUNT_IND = 0`. Price is the fallback reason (61% of notes when no discount).
- **Converted quotes carry the same reasons at the same rates** (e.g. price 51.6% vs 52.2%), and every note, even for converted quotes, has a "why not bound" clause. The reasons are generated *after* the fact from columns the model already has; they don't encode extra conversion logic.

## 2–3. Candidate library and residual screen
Correlation between each candidate's rank and the OOF residual (observed − predicted) of the 5-seed tuned CatBoost from `05_ensemble`. A feature carrying signal the model misses would show |r| above the noise level 2/√n ≈ 0.012.

In [3]:
oof = pd.read_parquet('final_oof.parquet')
assert (oof.POLICY_ID.values == L.POLICY_ID.values).all()
res = L[C.TARGET].to_numpy() - oof.oof_cb5.to_numpy()
rows = []
for c in V:
    x = L[c].astype(float); m = x.notna().to_numpy()
    r = x[m].rank().to_numpy()
    rows.append({'feature': c, 'r with residual': np.corrcoef((r - r.mean()) / r.std(), res[m])[0, 1], 'missing': 1 - m.mean()})
screen = pd.DataFrame(rows).assign(a=lambda t: t['r with residual'].abs()).sort_values('a', ascending=False).drop(columns='a')
noise = 2 / np.sqrt(len(L))
print(f'noise level {noise:.4f}; candidates above it: {(screen["r with residual"].abs() > noise).sum()} of {len(V)}')
screen.head(15).style.format({'r with residual': '{:+.4f}', 'missing': '{:.1%}'}).hide(axis='index')

noise level 0.0116; candidates above it: 0 of 67


feature,r with residual,missing
V2_PRICE_RANK_STATE,-0.0091,0.2%
V2_DRV_GT_VEH,-0.0090,0.0%
V2_VEH_AGE_STD,+0.0080,6.6%
V2_LOANED_MATCH,+0.0076,0.0%
V2_VEH_n_leased,-0.0076,0.0%
V2_VEH_SAFETY_MAX,-0.0071,0.0%
V2_PRICE_VS_STATE_PKG,-0.0071,0.2%
V2_DAY_OF_YEAR,-0.0069,0.0%
V2_VEH_ROWS_MINUS_TOTAL,-0.0067,0.0%
V2_LOG_PRICE,-0.0067,0.2%


**Reading:** **none of the 67 candidates** correlates with the residual above the noise level (largest |r| = 0.009 vs 0.012). The tuned CatBoost already extracts every main effect these features carry, including price vs. state/package peers, price ÷ ZIP income, the drivers > vehicles flag, and vehicle/driver composition counts.

## 4. Group tests in tuned CatBoost (seed-0 folds, paired Δ)
The residual screen only sees one feature at a time; a group could still help through interactions. Each group is added to the 41 features.

In [4]:
GROUPS = {
    'all 67 candidates': V,
    'driver counts/stats': [c for c in V if c.startswith('V2_DRV_')],
    'vehicle counts/stats': [c for c in V if c.startswith('V2_VEH_')],
    'drivers vs vehicles': ['V2_DRV_MINUS_VEH', 'V2_DRV_GT_VEH', 'V2_ADULTS_GT_VEH', 'V2_YOUNG_PER_VEH', 'V2_VEH_ROWS_MINUS_TOTAL',
                            'V2_LOANED_MATCH', 'V2_LEASED_MATCH', 'V2_MILES_PER_DRIVER', 'V2_COMMUTE_PER_VEH'],
    'objection count': ['V2_N_OBJECTIONS'],
    'price benchmarks': ['V2_LOG_PRICE', 'V2_PRICE_TO_INCOME', 'V2_PRICE_PER_VEH_ROW', 'V2_PRICE_VS_STATE_PKG', 'V2_PRICE_RANK_STATE'],
    'frequency + date': ['V2_FREQ_ZIP_CD', 'V2_FREQ_ZIP3', 'V2_DAY_OF_YEAR', 'V2_DAY_OF_MONTH', 'V2_DAYS_SINCE_START'],
}
KEY = {'all 67 candidates': 'all_v2', 'driver counts/stats': 'drv_counts', 'vehicle counts/stats': 'veh_counts', 'drivers vs vehicles': 'cross',
       'objection count': 'objections', 'price benchmarks': 'price', 'frequency + date': 'freq_date'}   # cache names from the scratch run
add = pd.DataFrame({g: test(f'v2probe_{KEY[g]}', F + cols) | {'n': len(cols)} for g, cols in GROUPS.items()}).T
add.style.format(fmt)

,auc,Δ auc,se Δ,n
all 67 candidates,0.7100,-0.0030,0.0006,67.000000
driver counts/stats,0.7107,-0.0023,0.0009,27.000000
vehicle counts/stats,0.7108,-0.0022,0.0005,23.000000
drivers vs vehicles,0.7113,-0.0017,0.0008,9.000000
objection count,0.7120,-0.0010,0.0006,1.000000
price benchmarks,0.7120,-0.0010,0.0008,5.000000
frequency + date,0.7110,-0.0020,0.0006,5.000000


## 5. Pruning: drop groups from the 41 features (tuned CatBoost, seed-0 folds)
Feature selection in `02` was done with untuned models. The tuned model is shallower and more regularized and may do better with less noise.

In [5]:
DROP = {
    'Census (12)': [c for c in F if c.startswith('ACS_')],
    'Census commute (6)': [c for c in F if c.startswith('ACS_COMMUTE')],
    'Census pop/land (3)': ['ACS_POPULATION', 'ACS_LAND_SQMI', 'ACS_POP_DENSITY'],
    'driver safety/accidents (4)': ['DRV_SAFETY_MIN', 'DRV_SAFETY_MEAN', 'DRV_ACC_FREE_MIN', 'DRV_ACC_FREE_MEAN'],
    'driver living/education (3)': ['DRV_OWN_SHARE', 'DRV_RENT_SHARE', 'DRV_HIGH_EDU_SHARE'],
    'driver commute (2)': ['DRV_COMMUTE_SUM', 'DRV_COMMUTE_MAX'],
    'age extras (3)': ['DRV_AGE_35_50', 'DRV_AGE_SPREAD', 'N_YOUNG_DRIVERS'],
    'age min/max (2)': ['DRV_AGE_MIN', 'DRV_AGE_MAX'],
    'young share (1)': ['DRV_YOUNG_SHARE'],
    'misc (year, luxury, price-missing)': ['QUOTE_YEAR', 'VEH_LUXURY_SHARE', 'QUOTED_AMT_MISSING'],
}
drop = pd.DataFrame({g: test(f'fe2_drop_{i}', [f for f in F if f not in cols]) | {'n': len(cols)} for i, (g, cols) in enumerate(DROP.items())}).T
drop.style.format(fmt)

,auc,Δ auc,se Δ,n
Census (12),0.7028,-0.0102,0.0009,11.000000
Census commute (6),0.7123,-0.0007,0.0008,5.000000
Census pop/land (3),0.7090,-0.0040,0.0011,3.000000
driver safety/accidents (4),0.7123,-0.0007,0.0010,4.000000
driver living/education (3),0.6997,-0.0133,0.0019,3.000000
driver commute (2),0.7110,-0.0020,0.0002,2.000000
age extras (3),0.7103,-0.0027,0.0015,3.000000
age min/max (2),0.7114,-0.0016,0.0003,2.000000
young share (1),0.7116,-0.0014,0.0006,1.000000
"misc (year, luxury, price-missing)",0.6946,-0.0184,0.0022,3.000000


In [6]:
# Confirm on all 15 folds: every drop that didn't hurt on seed 0 (Δ > -se), and their union. Seeds 1-2 are the unbiased read.
ok = [g for g in DROP if drop.loc[g, 'Δ auc'] > -drop.loc[g, 'se Δ']]
cands = {g: [f for f in F if f not in DROP[g]] for g in ok}
if len(ok) > 1:
    cands['union of passing drops'] = [f for f in F if not any(f in DROP[g] for g in ok)]
base15 = C.cv_score('fe2_base15', L, F, 'cb', P)
conf = []
for g, feats in cands.items():
    r = C.cv_score('fe2_conf_' + re.sub(r'\W+', '_', g), L, feats, 'cb', P)
    d = r['fold_aucs'] - base15['fold_aucs']
    conf.append({'variant': g, 'n_feats': len(feats), 'auc (15)': r['auc_mean'], 'Δ (15)': d.mean(), 'se (15)': d.std() / np.sqrt(15),
                 'Δ seeds 1-2': d[5:].mean(), 'se seeds 1-2': d[5:].std() / np.sqrt(10), 'top-decile': r['top_decile_prec'], 'feats': feats})
conf = pd.DataFrame(conf).set_index('variant') if conf else pd.DataFrame()
print(f'base 15-fold AUC {base15["auc_mean"]:.4f}')
conf.drop(columns='feats', errors='ignore').style.format({'auc (15)': '{:.4f}', 'Δ (15)': '{:+.4f}', 'se (15)': '{:.4f}',
                                                          'Δ seeds 1-2': '{:+.4f}', 'se seeds 1-2': '{:.4f}', 'top-decile': '{:.1%}'})

base 15-fold AUC 0.7118


,n_feats,auc (15),Δ (15),se (15),Δ seeds 1-2,se seeds 1-2,top-decile
variant,,,,,,,
Census commute (6),36,0.7117,-0.0001,0.0004,+0.0002,0.0004,30.0%
driver safety/accidents (4),37,0.7116,-0.0002,0.0004,+0.0001,0.0004,29.8%
union of passing drops,32,0.7122,+0.0005,0.0005,+0.0010,0.0004,29.9%


In [7]:
# Keep a pruned set only if it gains > 2 se on the unseen seeds 1-2.
win = conf[conf['Δ seeds 1-2'] > 2 * conf['se seeds 1-2']].sort_values('Δ seeds 1-2', ascending=False) if len(conf) else conf
FINAL_FEATS = win.iloc[0]['feats'] if len(win) else F
print('kept:', win.index[0] if len(win) else 'none - the 41 features stay', '|', len(FINAL_FEATS), 'features')
json.dump({'features': FINAL_FEATS, 'change': win.index[0] if len(win) else None}, open('features_v2.json', 'w'), indent=1)

kept: union of passing drops | 32 features


## Conclusions
- **Adding features doesn't help.** All 7 candidate groups lower seed-0 AUC (−0.001 to −0.003), and none of 67 features shows residual signal. The call notes add no new rules.
- **Pruning helps slightly.** Dropping Census commute shares (5), driver safety/accident aggregates (4) — 32 features — gives **+0.0010 ± 0.0004 on the unseen seeds 1–2** (+0.0005 on all 15 folds, AUC 0.7122). It's at the edge of the 2·se rule and saved to `features_v2.json`, but too small to be worth a leaderboard slot on its own.
- **What the model relies on** (drop tests): quote year / luxury / price-missing (−0.018 when dropped), driver living status & education (−0.013), Census density group (−0.010). These columns are strong, and their exact forms are what the generator uses.
- **Conclusion:** with these tables, the feature space looks exhausted at ~0.712 CV. The public-leaderboard gap (0.723 vs 0.738) is ~1 standard error of AUC on a few-thousand-row public sample, so it may be partly noise or leaderboard overfitting rather than a missing feature.